# FER2013 Training Experiments

This notebook is designed for Colab or Kaggle GPU training. It uses the dataset analysis from `01_data_exploration.ipynb` to justify the experiment protocol: a CNN trained from scratch for native 48x48 grayscale inputs, augmentation for low-resolution face variability, ResNet18 transfer learning for pretrained visual features, and imbalance-aware training for uneven emotion counts.


## 1. Setup

For Colab, clone your GitHub repository or upload it to Drive. For Kaggle, add the FER2013 image-folder dataset and set `DATA_DIR` to the Kaggle input path.


In [ ]:
# Optional in Colab/Kaggle if dependencies are missing:
# !pip install -q torch torchvision pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

# Local path after extracting Kaggle data:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'

# Kaggle example, uncomment and edit if needed:
# DATA_DIR = Path('/kaggle/input/fer2013')

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR


In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import torch

from fer_project.data import TransformConfig, build_imagefolder_dataloaders, class_weights, dataset_labels
import fer_project.metrics as metrics

metrics = importlib.reload(metrics)
collect_predictions = metrics.collect_predictions
plot_confusion_matrix = metrics.plot_confusion_matrix
save_classification_report = metrics.save_classification_report
top_confusions = metrics.top_confusions
from fer_project.models import build_model
from fer_project.training import fit

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


## 2. Experiment Configurations

The first five experiments are the core ablation study requested in the feedback. ResNet18 is the main pretrained baseline because it is a standard moderate-size transfer-learning model. MobileNetV2 is kept as an additional pretrained strategy because it tests a different question: whether a lightweight, efficient backbone can transfer useful features to FER2013 with fewer parameters and lower computational cost.


In [ ]:
CORE_EXPERIMENTS = [
    {
        'name': 'baseline_cnn',
        'group': 'required',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=False),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'baseline_cnn_aug',
        'group': 'required',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'resnet18_frozen',
        'group': 'required',
        'model_kind': 'transfer',
        'transfer_model': 'resnet18',
        'freeze_backbone': True,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-3,
        'epochs': 10,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'resnet18_finetune',
        'group': 'required',
        'model_kind': 'transfer',
        'transfer_model': 'resnet18',
        'freeze_backbone': False,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-4,
        'epochs': 10,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'baseline_cnn_aug_class_weights',
        'group': 'required',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': True,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
]

OPTIONAL_EXPERIMENTS = [
    {
        'name': 'baseline_cnn_aug_weighted_sampler',
        'group': 'optional_imbalance',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
        'weighted_sampler': True,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'baseline_cnn_aug_focal_loss',
        'group': 'optional_imbalance',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'focal',
        'focal_gamma': 2.0,
    },
    {
        'name': 'mobilenet_v2_frozen',
        'group': 'optional_transfer',
        'model_kind': 'transfer',
        'transfer_model': 'mobilenet_v2',
        'freeze_backbone': True,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-3,
        'epochs': 10,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
    {
        'name': 'mobilenet_v2_finetune',
        'group': 'optional_transfer',
        'model_kind': 'transfer',
        'transfer_model': 'mobilenet_v2',
        'freeze_backbone': False,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-4,
        'epochs': 10,
        'use_class_weights': False,
        'weighted_sampler': False,
        'loss_name': 'cross_entropy',
    },
]

# Set RUN_OPTIONAL_EXPERIMENTS = True when you have enough GPU time.
RUN_OPTIONAL_EXPERIMENTS = False
EXPERIMENTS = CORE_EXPERIMENTS + (OPTIONAL_EXPERIMENTS if RUN_OPTIONAL_EXPERIMENTS else [])


### Why ResNet18 and MobileNetV2?

The dataset analysis shows that FER2013 images are small, grayscale, imbalanced, and visually ambiguous. A custom CNN is useful because it matches the native 48x48 grayscale input and gives a fair from-scratch baseline. ResNet18 is the main pretrained model because it is a standard, moderate-size ImageNet backbone: strong enough to test whether generic visual features transfer to facial expressions, but still practical for this project.

MobileNetV2 is included as a second pretrained strategy, not as a replacement for ResNet18. Its purpose is different: it is a lightweight architecture designed for efficiency, so it lets the report compare a standard transfer backbone against a smaller transfer backbone. If MobileNetV2 performs close to ResNet18, that is useful evidence for computational efficiency; if it performs worse, that supports using the stronger ResNet18 model.

For both pretrained models, FER2013 images are converted from 48x48 grayscale to 224x224 RGB tensors with ImageNet normalization. Frozen experiments test whether fixed pretrained features are useful; fine-tuned experiments test whether adapting the whole backbone improves performance on FER2013.


## 3. Train One Experiment

Start with a tiny smoke test to confirm the code path works: one epoch, a small class-stratified subset, and `num_workers=0`. Then increase `subset_fraction` and epochs gradually. Transfer models must use the exact input adaptation justified in the data notebook: 48x48 grayscale FER2013 images are resized to 224x224, repeated into 3 channels, and normalized with ImageNet statistics for ResNet18.


In [ ]:
def run_experiment(config, batch_size=128, num_workers=0, subset_fraction=1.0):
    name = config['name']
    loaders, datasets = build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=config['train_config'],
        eval_config=config['eval_config'],
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=config.get('weighted_sampler', False),
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=42,
    )
    if config['model_kind'] == 'baseline_cnn':
        model = build_model('baseline_cnn')
    else:
        model = build_model(
            'transfer',
            transfer_model=config.get('transfer_model', 'resnet18'),
            freeze_backbone=config.get('freeze_backbone', True),
        )

    weights = class_weights(dataset_labels(datasets['train'])) if config.get('use_class_weights') else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit(
        model,
        loaders,
        device=DEVICE,
        epochs=config['epochs'],
        lr=config['lr'],
        class_weight=weights,
        loss_name=config.get('loss_name', 'cross_entropy'),
        focal_gamma=config.get('focal_gamma', 2.0),
        checkpoint_path=checkpoint_path,
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)

    report_path = RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv'
    report = save_classification_report(y_true, y_pred, report_path)
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)
    plot_confusion_matrix(y_true, y_pred, RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png')

    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    return history_frame, report, confusions


### Loss Curve Helper

Use this after each experiment to inspect convergence and possible overfitting from train/validation loss.


In [ ]:
def plot_loss_curve(history, title):
    ax = history.plot(
        x='epoch',
        y=['train_loss', 'val_loss'],
        marker='o',
        figsize=(5, 3),
        title=title,
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    plt.tight_layout()
    plt.show()
    return ax


## 4. Small Smoke Tests

Run these before the full experiment loop. They use one epoch and a small class-stratified subset, so they are only for checking that the dataloaders, model, training loop, metrics, and saved outputs work. Do not use these numbers as final results.


In [ ]:
# Fast CPU/GPU sanity check for the baseline CNN.
smoke_config = {**CORE_EXPERIMENTS[0], 'epochs': 1}
history, report, confusions = run_experiment(
    smoke_config,
    batch_size=32,
    num_workers=0,
    subset_fraction=0.05,
)
plot_loss_curve(history, 'baseline_cnn_smoke_test: train vs validation loss')
display(history[['epoch', 'train_loss', 'val_loss']])
display(report.loc[['macro avg', 'weighted avg']])
display(confusions)


In [ ]:
# Fast transfer-learning smoke test.
# Uses a tiny subset because ResNet18 resizes FER2013 images to 224x224.
transfer_smoke_config = {**CORE_EXPERIMENTS[2], 'epochs': 1}
history, report, confusions = run_experiment(
    transfer_smoke_config,
    batch_size=16,
    num_workers=0,
    subset_fraction=0.02,
)
plot_loss_curve(history, 'resnet18_frozen_smoke_test: train vs validation loss')
display(history[['epoch', 'train_loss', 'val_loss']])
display(report.loc[['macro avg', 'weighted avg']])
display(confusions)


## 5. Run Experiments Safely

Do not run every configuration in one fragile loop unless you really want to. The cells below let you run a single experiment, then save or merge results incrementally. This is safer in Colab/Kaggle because if one model fails or the session disconnects, the completed experiment outputs are already saved under `results/`.


In [ ]:
EXPERIMENTS_BY_NAME = {config['name']: config for config in CORE_EXPERIMENTS + OPTIONAL_EXPERIMENTS}

# Recommended workflow: change this name and run one experiment at a time.
SELECTED_EXPERIMENT = 'baseline_cnn'

history, report, confusions = run_experiment(EXPERIMENTS_BY_NAME[SELECTED_EXPERIMENT])
plot_loss_curve(history, f'{SELECTED_EXPERIMENT}: train vs validation loss')
display(history[['epoch', 'train_loss', 'val_loss']])
display(report.loc[['macro avg', 'weighted avg']])
display(report.loc[['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'], ['precision', 'recall', 'f1-score', 'support']])
display(confusions)


### Run A Small Group

Use this when you want to compare a controlled subset, such as only baseline CNN experiments or only transfer-learning experiments. Keep groups small so failures are easy to recover from.


In [ ]:
EXPERIMENT_GROUPS = {
    'core_baselines': ['baseline_cnn', 'baseline_cnn_aug', 'baseline_cnn_aug_class_weights'],
    'resnet18_transfer': ['resnet18_frozen', 'resnet18_finetune'],
    'imbalance_optional': ['baseline_cnn_aug_weighted_sampler', 'baseline_cnn_aug_focal_loss'],
    'mobilenet_transfer': ['mobilenet_v2_frozen', 'mobilenet_v2_finetune'],
    'core_required': [config['name'] for config in CORE_EXPERIMENTS],
}

SELECTED_GROUP = 'core_baselines'
group_results = {}

for experiment_name in EXPERIMENT_GROUPS[SELECTED_GROUP]:
    history, report, confusions = run_experiment(EXPERIMENTS_BY_NAME[experiment_name])
    plot_loss_curve(history, f'{experiment_name}: train vs validation loss')

    group_results[experiment_name] = {
        'group': EXPERIMENTS_BY_NAME[experiment_name]['group'],
        'final_val_loss': float(history['val_loss'].iloc[-1]),
        'best_val_loss': float(history['val_loss'].min()),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
        'top_confusion': 'none' if confusions.empty else f"{confusions.iloc[0]['true_emotion']} -> {confusions.iloc[0]['predicted_emotion']}",
    }

group_summary = pd.DataFrame(group_results).T.sort_values('test_macro_f1', ascending=False)
group_summary


### Merge Finished Results

Use this only after you have finished several separate runs. It reads saved metric files and displays one comparison table without retraining. The purpose is convenience, not replacing the visible F1/loss outputs above.


In [ ]:
summary_rows = {}
metrics_dir = RESULTS_DIR / 'metrics'

for config in CORE_EXPERIMENTS + OPTIONAL_EXPERIMENTS:
    name = config['name']
    report_path = metrics_dir / f'{name}_classification_report.csv'
    history_path = metrics_dir / f'{name}_history.csv'
    confusions_path = metrics_dir / f'{name}_top_confusions.csv'
    if not report_path.exists() or not history_path.exists():
        continue

    report = pd.read_csv(report_path, index_col=0)
    history = pd.read_csv(history_path)
    confusions = pd.read_csv(confusions_path) if confusions_path.exists() else pd.DataFrame()
    summary_rows[name] = {
        'group': config['group'],
        'final_val_loss': float(history['val_loss'].iloc[-1]),
        'best_val_loss': float(history['val_loss'].min()),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
        'top_confusion': 'none' if confusions.empty else f"{confusions.iloc[0]['true_emotion']} -> {confusions.iloc[0]['predicted_emotion']}",
    }

summary = pd.DataFrame(summary_rows).T.sort_values('test_macro_f1', ascending=False)
summary.to_csv(metrics_dir / 'experiment_summary.csv')
summary


## 6. Report Checklist

- Start the report from dataset evidence: 48x48 grayscale images, class imbalance, visual ambiguity, and train/test split sizes.
- Treat `CORE_EXPERIMENTS` as the minimum protocol for the instructor feedback.
- Explain why ResNet18 is the main pretrained backbone: standard, moderate-size, ImageNet-pretrained, and practical for frozen-vs-fine-tuned transfer learning.
- Explain why MobileNetV2 is added: a lightweight pretrained backbone for efficiency-oriented comparison against ResNet18.
- If time allows, run `OPTIONAL_EXPERIMENTS` to strengthen the imbalance and transfer-learning sections.
- Select the best model by macro F1 because FER2013 is imbalanced.
- Use weighted F1 and per-class precision/recall/F1 as supporting metrics.
- Do not use accuracy for model selection or report conclusions.
- Use confusion matrices and each `_top_confusions.csv` file to discuss confused emotions and failure cases.
- Explain the transfer learning input adaptation: 48x48 grayscale images are converted to 3-channel 224x224 tensors with ImageNet normalization.
- Discuss which modeling choice improved minority or ambiguous classes, even if total accuracy did not improve.
